# Part 3 — Microcensus Data Preparation

Objective: Filter residents of the Canton of Neuchâtel using household coordinates, retaining only survey reference days from Monday through Friday including respondents who were at home all day. MATSim is not processed here, nor are activity chain comparisons performed.

Data sources: `microcensus/haushalte.csv`, `zielpersonen.csv`, `wege.csv`; boundary: `Boundary/neuchatel.shp`. Field coding follows the MZMV2015 Excel data dictionary in the same directory.

**Definitions**: `tag` refers to the day of the week on the survey reference date (Stichtag), not the date of the telephone interview; 1–5 correspond to Monday through Friday. `f50200`: 1 = out of the home, 2 = at home. Household coordinates in CH1903 / LV03 (EPSG:21781) must be converted to the boundary coordinate system. Points on the boundary are included. Invalid coordinates are reported separately; do not use canton codes as a substitute for spatial filtering. The original CSV files remain unmodified.

Dependencies: pandas, geopandas, pyproj, shapely (installed in the selected Python kernel).

In [1]:
from pathlib import Path
import pandas as pd
import geopandas as gpd
from IPython.display import display

# You can run it from the project directory or any of its subdirectories.
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "microcensus/haushalte.csv").exists()), None)
if ROOT is None:
    raise FileNotFoundError("Please launch the notebook from the `bicycle` project directory or one of its subdirectories.")
DATA = ROOT / "microcensus"
BOUNDARY = ROOT / "Boundary/neuchatel.shp"
CSV_OPTIONS = dict(encoding="cp1252", low_memory=False)

households = pd.read_csv(DATA / "haushalte.csv", **CSV_OPTIONS)
respondents = pd.read_csv(DATA / "zielpersonen.csv", **CSV_OPTIONS)
assert households["HHNR"].is_unique, "The household ID is duplicated; you must first check the associated records."
assert respondents["HHNR"].is_unique, "This process assumes one target respondent per household."
assert respondents["HHNR"].isin(households["HHNR"]).all()
print(f"Households: {len(households):,}; Target respondents: {len(respondents):,}")


Households: 57,090; Target respondents: 57,090


## 1. Household Coordinate Filtering

Use `W_X_CH1903` / `W_Y_CH1903`; do not directly mix LV03 and LV95 coordinates. The broad-range checks below are used only to filter out missing values and special codes; the final determination of whether a location belongs within the canton is made based on the polygon. `W_KANTON` is used for auditing purposes only.

In [2]:
boundary = gpd.read_file(BOUNDARY)
assert boundary.crs is not None, "The boundary is missing a CRS"
assert boundary.geometry.notna().all() and boundary.geometry.is_valid.all()
boundary_shape = boundary.geometry.union_all()
x = pd.to_numeric(households["W_X_CH1903"], errors="coerce")
y = pd.to_numeric(households["W_Y_CH1903"], errors="coerce")
valid_home = x.between(400_000, 900_000) & y.between(0, 400_000)
home_points = gpd.GeoDataFrame(
    households.loc[valid_home, ["HHNR"]].copy(),
    geometry=gpd.points_from_xy(x[valid_home], y[valid_home]),
    crs="EPSG:21781",
).to_crs(boundary.crs)
home_points["inside_boundary"] = home_points.geometry.intersects(boundary_shape)
resident_ids = home_points.loc[home_points["inside_boundary"], "HHNR"]
households_ne = households.loc[households["HHNR"].isin(resident_ids)].copy()
invalid_home_coordinates = households.loc[~valid_home,
    ["HHNR", "W_X_CH1903", "W_Y_CH1903", "W_KANTON"]].copy()
assert len(households_ne) > 0
print("Boundary CRS: ", boundary.crs)
print(f"Invalid coordinates: {len(invalid_home_coordinates):,}; Households within the canton: {len(households_ne):,}")
display(households_ne["W_KANTON"].value_counts(dropna=False).rename("households").to_frame())


Boundary CRS:  COMPD_CS["CH1903+ / LV95 + LN02 height",PROJCS["CH1903+ / LV95",GEOGCS["CH1903+",DATUM["CH1903+",SPHEROID["Bessel 1841",6377397.155,299.1528128,AUTHORITY["EPSG","7004"]],AUTHORITY["EPSG","6150"]],PRIMEM["Greenwich",0,AUTHORITY["EPSG","8901"]],UNIT["degree",0.0174532925199433,AUTHORITY["EPSG","9122"]],AUTHORITY["EPSG","4150"]],PROJECTION["Hotine_Oblique_Mercator_Azimuth_Center"],PARAMETER["latitude_of_center",46.9524055555556],PARAMETER["longitude_of_center",7.43958333333333],PARAMETER["azimuth",90],PARAMETER["rectified_grid_angle",90],PARAMETER["scale_factor",1],PARAMETER["false_easting",2600000],PARAMETER["false_northing",1200000],UNIT["metre",1,AUTHORITY["EPSG","9001"]],AXIS["Easting",EAST],AXIS["Northing",NORTH],AUTHORITY["EPSG","2056"]],VERT_CS["LN02 height",VERT_DATUM["Landesnivellement 1902",2005,AUTHORITY["EPSG","5127"]],UNIT["metre",1,AUTHORITY["EPSG","9001"]],AXIS["Gravity-related height",UP],AUTHORITY["EPSG","5728"]]]
Invalid coordinates: 0; Households within t

,households
W_KANTON,
24,2837


## 2. Weekday Filtering

Retain the complete personal records and verify that the calendar days of the week for `tag` and `VSTag` match. Filter based on the reference date; do not exclude public holidays that fall on weekdays.

In [3]:
reference_date = pd.to_datetime(respondents["VSTag"], format="%m/%d/%Y", errors="coerce")
weekday_code = pd.to_numeric(respondents["tag"], errors="coerce")
date_mismatch = reference_date.notna() & weekday_code.ne(reference_date.dt.dayofweek + 1)
assert not date_mismatch.any(), "The tag does not match the reference date. Please check."
respondents_ne = respondents.loc[respondents["HHNR"].isin(resident_ids)].copy()
microcensus_persons = respondents_ne.loc[respondents_ne["tag"].isin([1, 2, 3, 4, 5])].copy()
# Add the "hh_" prefix to all family fields to avoid overwriting fields with the same name in the individual table.
hh_details = households_ne.rename(columns={c: f"hh_{c}" for c in households_ne if c != "HHNR"})
microcensus_persons = microcensus_persons.merge(hh_details, on="HHNR", how="left", validate="one_to_one")
assert len(microcensus_persons) > 0
display(microcensus_persons["tag"].value_counts().sort_index().rename("respondents").to_frame())


<ipython-input-1-b0bb04201516>:3: Pandas4Warning: Series.dt.dayofweek is deprecated and will be removed in a future version. Use Series.dt.day_of_week instead.
  date_mismatch = reference_date.notna() & weekday_code.ne(reference_date.dt.dayofweek + 1)


,respondents
tag,
1,486
2,427
3,376
4,379
5,334


## 3. Retain records for individuals who did not go out, and cross-check them against trip records

Based on the individual table, perform a left join with the trip count. Only when `f50200=2` and there are no trips should the record be marked as `at home all day`; unknown or conflicting records should be retained and marked as "pending review." Do not assume someone was at home all day solely because there are no trips. This label only indicates that the individual did not go out that day; it does not imply working from home.

The trip table is filtered only by the selected individual; cross-canton trips are not removed. `trip_count` represents the total number of complete `wege` records, not the number of segments.

In [4]:
selected_ids = set(microcensus_persons["HHNR"])
trip_parts = []
for chunk in pd.read_csv(DATA / "wege.csv", chunksize=50_000, **CSV_OPTIONS):
    trip_parts.append(chunk.loc[chunk["HHNR"].isin(selected_ids)].copy())
microcensus_trips = pd.concat(trip_parts, ignore_index=True)
assert not microcensus_trips.duplicated(["HHNR", "WEGNR"]).any()
trip_counts = microcensus_trips.groupby("HHNR").size()
microcensus_persons["trip_count"] = microcensus_persons["HHNR"].map(trip_counts).fillna(0).astype(int)
reported = microcensus_persons["f50200"]
counts = microcensus_persons["trip_count"]
microcensus_persons["day_status"] = "unknown / check"
microcensus_persons.loc[reported.eq(2) & counts.eq(0), "day_status"] = "at home all day"
microcensus_persons.loc[reported.eq(1) & counts.gt(0), "day_status"] = "travelled"
conflict = (reported.eq(2) & counts.gt(0)) | (reported.eq(1) & counts.eq(0))
microcensus_persons.loc[conflict, "day_status"] = "inconsistent / check"
microcensus_persons["at_home_all_day"] = microcensus_persons["day_status"].eq("at home all day")
review_cases = microcensus_persons.loc[~microcensus_persons["day_status"].isin(
    ["travelled", "at home all day"]), ["HHNR", "f50200", "trip_count", "day_status"]].copy()
assert len(microcensus_persons) == len(respondents_ne.loc[respondents_ne["tag"].isin(range(1, 6))])
assert microcensus_persons["f50200"].eq(2).sum() == respondents_ne.loc[
    respondents_ne["tag"].isin(range(1, 6)), "f50200"].eq(2).sum()
assert microcensus_persons["trip_count"].sum() == len(microcensus_trips)
assert microcensus_trips["HHNR"].isin(selected_ids).all()
assert microcensus_persons["WP"].notna().all() and microcensus_persons["WP"].gt(0).all()


## 4. Filtering Audit and Subsequent Available Data

The number of persons represents the unweighted sample size; proportions are calculated separately using the original individual weights `WP`. The sum of the weights is not interpreted here as the total cantonal population, nor is it multiplied by 10.

- `microcensus_persons`: Canton residents whose survey reference day was a weekday; retains the original fields, individual weights, household fields, and travel status.
- `microcensus_trips`: Complete trip records for these respondents.
- `households_ne`: Households within the canton (not yet filtered by respondents’ reference weekdays).
- `review_cases` / `invalid_home_coordinates`: Records with status conflicts or invalid coordinates awaiting verification.

Currently, only data preparation is being performed; unknown statuses are not forcibly classified, and original data is not exported or overwritten.

In [5]:
filter_audit = pd.DataFrame({
    "stage": ["All respondents", "Residents inside boundary", "Residents on weekdays",
              "Reported non-travellers retained", "At home all day (confirmed)", "Review cases", "Selected trips"],
    "count": [len(respondents), len(respondents_ne), len(microcensus_persons),
              int(reported.eq(2).sum()), int(microcensus_persons["at_home_all_day"].sum()),
              len(review_cases), len(microcensus_trips)]
})
status_summary = microcensus_persons.groupby("day_status").agg(
    sample_count=("HHNR", "size"), weight_sum=("WP", "sum"))
status_summary["sample_share"] = status_summary["sample_count"] / len(microcensus_persons)
status_summary["weighted_share"] = status_summary["weight_sum"] / microcensus_persons["WP"].sum()
display(filter_audit)
display(status_summary)
print("Records pending review:", len(review_cases))
if len(review_cases):
    display(review_cases.head(10))
print("Validation passed: Non-travelers retained, no duplicates in associations, trip counts match, and weights are valid.")


Records pending review: 0
Validation passed: Non-travelers retained, no duplicates in associations, trip counts match, and weights are valid.


,stage,count
0,All respondents,57090
1,Residents inside boundary,2837
2,Residents on weekdays,2002
3,Reported non-travellers retained,170
4,At home all day (confirmed),170
5,Review cases,0
6,Selected trips,7835


,sample_count,weight_sum,sample_share,weighted_share
day_status,,,,
at home all day,170,73.423438,0.084915,0.084327
travelled,1832,797.275423,0.915085,0.915673


## 5. Export filtered data
Export the selected respondents and their trips to `microcensus_filter/` as UTF-8 CSV files without a dataframe index. Non-travelers remain in the persons file. Join the files using `HHNR`. Original source files remain unchanged.


In [6]:
EXPORT_DIR = ROOT / "microcensus_filter"
EXPORT_DIR.mkdir(parents=True, exist_ok=True)
persons_path = EXPORT_DIR / "microcensus_ne_weekday_persons.csv"
trips_path = EXPORT_DIR / "microcensus_ne_weekday_trips.csv"
microcensus_persons.to_csv(persons_path, index=False, encoding="utf-8")
microcensus_trips.to_csv(trips_path, index=False, encoding="utf-8")

# Read back the exports and verify population coverage and trip counts.
exported_persons = pd.read_csv(persons_path, encoding="utf-8", low_memory=False)
exported_trips = pd.read_csv(trips_path, encoding="utf-8", low_memory=False)
assert len(exported_persons) == len(microcensus_persons)
assert len(exported_trips) == len(microcensus_trips)
assert exported_persons["HHNR"].is_unique
assert exported_trips["HHNR"].isin(exported_persons["HHNR"]).all()
assert exported_persons["at_home_all_day"].sum() == microcensus_persons["at_home_all_day"].sum()
assert exported_persons["trip_count"].sum() == len(exported_trips)
assert exported_persons.columns.tolist() == microcensus_persons.columns.tolist()
assert exported_trips.columns.tolist() == microcensus_trips.columns.tolist()
print(f"Exported {len(exported_persons):,} respondents to {persons_path.name}")
print(f"Exported {len(exported_trips):,} trips to {trips_path.name}")
print("Export validation passed.")


Exported 2,002 respondents to microcensus_ne_weekday_persons.csv
Exported 7,835 trips to microcensus_ne_weekday_trips.csv
Export validation passed.
